# Imports

In [1]:
!unzip src.zip

/bin/bash: line 1: unzip: command not found


In [2]:
import sys
# sys.path.append('src') # colab testing
sys.path.append('../src') # local testing

import pandas as pd
from transformers import Trainer
from copy import deepcopy


/home/damian/miniconda3/envs/amyloid/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pip install Bio

Note: you may need to restart the kernel to use updated packages.


In [4]:
from deployment import Integrator
from deployment import Preprocessor
from deployment import train_model

/home/damian/miniconda3/envs/amyloid/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/damian/miniconda3/envs/amyloid/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened

Imported model building :)


# Data processing


In [5]:
raw_file_path = "../amyloid_sheet_16Oct.csv"
raw_df = pd.read_csv(raw_file_path)
raw_df

,PMID,Rejection?,If so; reason to reject?,Other? Expand,No access to full-text,Decided by what?,URL,Year,Year.1,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,39410666,Useful,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/39410666/,NaN,NaN,NaN,NaN,NaN,NaN,Open questions:,NaN,NaN,NaN,NaN,NaN,NaN
1,39363348,Useful,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/39363348/,NaN,NaN,NaN,NaN,NaN,NaN,1) How do we consider an antibody used to dete...,NaN,Not enough experimental data. Same for isolati...,NaN,NaN,NaN,NaN
2,39358820,Useful,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/39358820/,NaN,NaN,NaN,NaN,NaN,NaN,2) What about Pre-prints?,NaN,No to pre-prints,NaN,NaN,NaN,NaN
3,39350371,Useful,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/39350371/,NaN,NaN,NaN,NaN,NaN,NaN,3) Non-English papers? I would trust automatic...,NaN,No to Non-Englishs,NaN,NaN,NaN,NaN
4,39319998,Rejected,Not enough experimental data,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/39319998/,NaN,NaN,NaN,NaN,NaN,NaN,4) There are doubles. Excluding them?,NaN,Remove them,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4018,19664624,NaN,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/19664624/,2010,Jun,19664624,FALSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4019,19659869,NaN,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/19659869/,Loading...,Loading...,19659869,FALSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4020,19647749,NaN,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/19647749/,#REF!,#REF!,19647749,FALSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4021,19636575,NaN,NaN,NaN,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/19636575/,#REF!,#REF!,19636575,FALSE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
integrator = Integrator(raw_path = raw_file_path, pmid_col = "PMID", email = "test@gmail.com")
# email is used for Entrez API and is not necessary

integrator.reduce_columns(keep_columns = ['PMID', "Rejection?", "If so; reason to reject?"])
integrator.fetch_pubmed(save_path="/home/damian/projects/Deepskim/amyloid_sheet_16Oct_new.csv")
integrator.merge()



dataset already available at /home/damian/projects/Deepskim/amyloid_sheet_16Oct_new.csv - skipping fetching from pubmed and loading directly


In [11]:
merged = integrator.merged_df.copy()
preprocessor = Preprocessor(merged)

preprocessor.dropna(subset=["Abstract"])
preprocessor.drop_values(column="If so; reason to reject?", value="Review article")
preprocessor.map_labels(label_col="Rejection?", mapping={"Rejected": 0, "Useful": 1})
preprocessor.split_dataset(label_col="Rejection?")

Dropped 61 rows containing NaN. Dropped rows are stored in self.dropped_df.
Dropped 809 rows (condition: If so; reason to reject? == Review article). Dropped rows are stored in self.dropped_df.
Mapped labels in column 'Rejection?' using provided mapping. Value counts: {0: 1754, -1: 1232, 1: 167}. Unmapped (set to -1): 1232 rows.
Dataset split completed: 1921 rows in self.train_df, 1232 rows in self.test_df. Data stored in these attributes.


In [8]:
train = preprocessor.train_df
train.to_csv("train.csv", index=False)
test = preprocessor.test_df
test.to_csv("test.csv", index=False)

# Training

In [17]:
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"

DEFAULT_CONFIG = {
    "seed": 1,
    "device": "cuda",  # "cuda" or "cpu"
    "wandb_project_name": "Amyloid-test",
    "data": {
        "train_file_path": "train.csv",
        "test_file_path": "test.csv",
        "target": "Rejection?",
        "test_size": 0.15,
        "model_name": model_name,
        "max_length": 380,
        "keywords": ["aggregates", "amyloid", "scfv", 'hiapp', 'mab', 'ttr', 'donanemab', 'aggregation'],
        "special_tokens": ["[KEY]", "[J_END]", "[T_END]"],
        "seed": 1,
    },
    "model": {
        "model_name": model_name,
        "num_labels": 2,
        "special_tokens": ['[KEY]', '[J_END]', '[T_END]'],
        "unfreeze_last_k_layers": 12,
        "change_classifier": False
    },
    "training": {
        "hparams": {
            "learning_rate": 3e-5,
            "num_train_epochs": 10,
            "per_device_train_batch_size": 16,
            "weight_decay": 0.1,
            "classifier_hidden_dim": 512,
            "warmup_ratio": 0.1
        }
    },
}

In [10]:
import wandb
wandb.login(key="36df08bdc35c9269bbc0cf195e6a4f4c3d318dac")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/damian/.netrc
wandb: Currently logged in as: damian1150 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [18]:
artifacts, hf_test = train_model(DEFAULT_CONFIG)

Using device: cuda


2025-10-16 10:47:44,612 - INFO - Splitting data using the selected strategy.
2025-10-16 10:47:44,613 - INFO - Performing stratified train-test split.
2025-10-16 10:47:44,618 - INFO - Train-test split completed.
Map: 100%|██████████| 291/291 [00:00<00:00, 2631.92 examples/s]
2025-10-16 10:47:45,292 - INFO - Converted train-test split into a single DatasetDict with 'train' and 'test' Datasets.
2025-10-16 10:47:45,293 - INFO - Train-val split: Train=1648, Val=291
Map: 100%|██████████| 1241/1241 [00:00<00:00, 2991.74 examples/s]
2025-10-16 10:47:45,727 - INFO - Converted DataFrame into DatasetDict with single split 'test'. has_labels=False
2025-10-16 10:47:45,728 - INFO - Test dataset size: Test=1241
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use 

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.496800,0.501238,0.886598,0.382353,0.520000,0.440678
2,0.328500,0.602127,0.883162,0.378378,0.560000,0.451613
3,0.511300,0.956024,0.927835,0.625000,0.400000,0.487805
4,0.220100,1.274026,0.938144,0.733333,0.440000,0.550000
5,0.001900,1.229606,0.910653,0.483871,0.600000,0.535714
6,0.000300,1.622902,0.920962,0.538462,0.560000,0.549020
7,0.002500,1.634367,0.914089,0.500000,0.560000,0.528302
8,0.000100,1.692919,0.914089,0.500000,0.560000,0.528302
9,0.000100,1.723559,0.914089,0.500000,0.560000,0.528302
10,0.000100,1.730449,0.914089,0.500000,0.560000,0.528302


best_train_accuracy,▁
best_train_f1,▁
best_train_loss,▁
best_train_model_preparation_time,▁
best_train_precision,▁
best_train_recall,▁
best_train_runtime,▁
best_train_samples_per_second,▁
best_train_steps_per_second,▁
best_val_accuracy,▁
+24,...


# Prediction

In [12]:
trainer = artifacts["trainer"]
eval_args = deepcopy(trainer.args)
eval_args.report_to = []
eval_args.eval_strategy = "no"

eval_trainer = Trainer(
    model=trainer.model,
    args=eval_args,
    compute_metrics=trainer.compute_metrics,
)

hf_test.set_format(type="torch", columns=["input_ids", "attention_mask"])
predictions_output = eval_trainer.predict(hf_test['test'])
predicted_labels = predictions_output.predictions.argmax(axis=1)

hf_test.reset_format()
pmids = hf_test['test']["PMID"]
results_df = pd.DataFrame({
    "PMID": pmids,
    "predicted_label": predicted_labels,
})


In [13]:
print(results_df['predicted_label'].value_counts())
results_df


predicted_label
0    1222
1      19
Name: count, dtype: int64


,PMID,predicted_label
0,35133388,0
1,28800329,0
2,25115543,1
3,24194640,0
4,30585717,1
...,...,...
1236,19664624,0
1237,19659869,0
1238,19647749,0
1239,19636575,0


In [14]:
positives = results_df[results_df['predicted_label']==1]
positives

,PMID,predicted_label
2,25115543,1
4,30585717,1
9,33167834,1
12,29298867,1
118,27347598,1
155,27029347,1
223,26493635,1
312,25933020,1
409,25352592,1
417,25262917,1


# Save

In [15]:
results_df.to_csv("results.csv", index=False) # save results
positives.to_csv("positives.csv", index=False)

